<a href="https://colab.research.google.com/github/miriamamin1213-ux/best_models/blob/main/GPT2_tabular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q gdown

import gdown

gdown.download(
    id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
    output="HPV2025.xlsx",
    quiet=False
)

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

from transformers import GPT2Model


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# Load and preprocess data
# ============================================================

df=pd.read_excel("HPV2025.xlsx")

df = df.dropna(subset=["HPV Status"])

tobacco_mode = df["Tobacco Consumption"].mode()[0]
df["Tobacco Consumption"] = (
    df["Tobacco Consumption"].fillna(tobacco_mode)
)

alcohol_mode = df["Alcohol Consumption"].mode()[0]
df["Alcohol Consumption"] = (
    df["Alcohol Consumption"].fillna(alcohol_mode)
)

df = df.drop(
    columns=[
        "PatientID",
        "CenterID",
        "Task 1",
        "Task 2",
        "Task 3"
    ]
)

df = df.dropna()

df["T-stage"] = df["T-stage"].replace(
    {
        "T0": 0,
        "T1": 1,
        "T2": 2,
        "T3": 3,
        "T4": 4
    }
)

df["N-stage"] = df["N-stage"].replace(
    {
        "N0": 0,
        "N1": 1,
        "N2": 2,
        "N3": 3
    }
)

df["M-stage"] = df["M-stage"].replace(
    {
        "M0": 0,
        "M1": 1
    }
)

feature_columns = [
    "Age",
    "Gender",
    "Tobacco Consumption",
    "Alcohol Consumption",
    "Performance Status",
    "Relapse",
    "RFS",
    "Treatment",
    "T-stage",
    "N-stage",
    "M-stage"
]

X = df[feature_columns].copy()
y = df["HPV Status"].copy()

print("Dataset shape:", df.shape)


# ============================================================
# Train-test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Train samples:")
print(y_train.value_counts())

print("\nTest samples:")
print(y_test.value_counts())


# ============================================================
# Standardisation
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ============================================================
# SMOTE
# ============================================================

smote = SMOTE(random_state=SEED)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nTraining samples after SMOTE:")
print(pd.Series(y_train_smote).value_counts())


# ============================================================
# Convert to tensors
# ============================================================

X_train_smote = torch.tensor(
    X_train_smote,
    dtype=torch.float32
)

y_train_smote = torch.tensor(
    np.asarray(y_train_smote),
    dtype=torch.long
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.long
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256

train_dataset = TensorDataset(
    X_train_smote,
    y_train_smote
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT-2 model for numerical tabular features
# ============================================================

class HPVNetGPT2Numerical(nn.Module):

    def __init__(
        self,
        num_features=11,
        num_classes=2,
        model_name="openai-community/gpt2",
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.num_features = num_features
        self.freeze_gpt2 = freeze_gpt2

        # Load pretrained GPT-2 weights
        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False

        hidden_size = self.gpt2.config.hidden_size

        # Map each scalar value into GPT-2 embedding space
        self.feature_projection = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size)
        )

        # Learnable identity embedding for each clinical feature
        self.feature_embedding = nn.Parameter(
            torch.empty(
                1,
                num_features,
                hidden_size
            )
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        nn.init.normal_(
            self.feature_embedding,
            mean=0.0,
            std=0.02
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        # Disable GPT-2 dropout when GPT-2 is frozen
        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(self, x):
        # x: [batch_size, 11]

        # [batch_size, 11, 1]
        x = x.unsqueeze(-1)

        # [batch_size, 11, hidden_size]
        x = self.feature_projection(x)

        # Add feature identity embeddings
        x = x + self.feature_embedding

        outputs = self.gpt2(
            inputs_embeds=x,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Mean pooling over the 11 feature tokens
        pooled_output = hidden_states.mean(dim=1)

        logits = self.classifier(pooled_output)

        return logits


# ============================================================
# Model, loss and optimiser
# ============================================================

model = HPVNetGPT2Numerical(
    num_features=len(feature_columns),
    num_classes=2,
    freeze_gpt2=True
).to(device)

weights = torch.tensor(
    [3.0, 1.0],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 100
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(
            device,
            non_blocking=True
        )

        batch_y = batch_y.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(batch_x)

        train_loss = criterion(
            outputs,
            batch_y
        )
        print('epoch:',epoch, 'loss:',train_loss.item())
        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item() * batch_x.size(0)
        )

        train_sample_count += batch_x.size(0)

    average_train_loss = (
        train_loss_sum / train_sample_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():

        for batch_x, batch_y in test_loader:

            batch_x = batch_x.to(
                device,
                non_blocking=True
            )

            batch_y = batch_y.to(
                device,
                non_blocking=True
            )

            test_outputs = model(batch_x)

            test_loss = criterion(
                test_outputs,
                batch_y
            )

            test_loss_sum += (
                test_loss.item() * batch_x.size(0)
            )

            test_sample_count += batch_x.size(0)

    average_test_loss = (
        test_loss_sum / test_sample_count
    )

    if average_test_loss < best_val_loss:

        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_numerical.pth"
        )

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Inference
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_numerical.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():

    for batch_x, batch_y in test_loader:

        batch_x = batch_x.to(
            device,
            non_blocking=True
        )

        outputs = model(batch_x)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu()
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            batch_y.cpu()
        )


probabilities = torch.cat(
    all_probabilities,
    dim=0
).numpy()

predicted = torch.cat(
    all_predictions,
    dim=0
).numpy()

y_true = torch.cat(
    all_targets,
    dim=0
).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:, 1]
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 39.3MB/s]


Using device: cuda


/tmp/ipykernel_3692/1394420336.py:106: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["T-stage"] = df["T-stage"].replace(
/tmp/ipykernel_3692/1394420336.py:116: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["N-stage"] = df["N-stage"].replace(
/tmp/ipykernel_3692/1394420336.py:125: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_s

Dataset shape: (423, 12)
Train samples:
HPV Status
1.0    320
0.0     18
Name: count, dtype: int64

Test samples:
HPV Status
1.0    80
0.0     5
Name: count, dtype: int64

Training samples after SMOTE:
HPV Status
1.0    320
0.0    320
Name: count, dtype: int64


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Trainable parameters: 111746
epoch: 0 loss: 0.5970104336738586


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


epoch: 0 loss: 0.5849360823631287
epoch: 0 loss: 0.5525439381599426
epoch: 1 loss: 0.6041127443313599
epoch: 1 loss: 0.5357102751731873
epoch: 1 loss: 0.5758432745933533
epoch: 2 loss: 0.573809802532196
epoch: 2 loss: 0.549385666847229
epoch: 2 loss: 0.5465991497039795
epoch: 3 loss: 0.5526176691055298
epoch: 3 loss: 0.5551117658615112
epoch: 3 loss: 0.5619131922721863
epoch: 4 loss: 0.5565614700317383
epoch: 4 loss: 0.5589823722839355
epoch: 4 loss: 0.49488797783851624
epoch: 5 loss: 0.5361388921737671
epoch: 5 loss: 0.5268328785896301
epoch: 5 loss: 0.5482576489448547
epoch: 6 loss: 0.5381160974502563
epoch: 6 loss: 0.5161407589912415
epoch: 6 loss: 0.5040193796157837
epoch: 7 loss: 0.4959838092327118
epoch: 7 loss: 0.5063904523849487
epoch: 7 loss: 0.5385801196098328
epoch: 8 loss: 0.4905129373073578
epoch: 8 loss: 0.5043231844902039
epoch: 8 loss: 0.4686559736728668
epoch: 9 loss: 0.508521556854248
epoch: 9 loss: 0.4156782031059265
epoch: 9 loss: 0.5184917449951172
Epoch 010, Train